# Step 3: Attention

## 3.1 Ne Değişiyor?

Önceki model, sonraki tokenı yalnızca mevcut tokena göre
tahmin ediyordu. Yeni model, konum bilgisini ve önceki
tokenlardan gelen bilgileri de kullanacak.

Bu bölümde konum embeddingleri, RMSNorm, tek başlı attention,
residual bağlantılar, çıktı katmanı ve KV cache eklenecek.
Veri kümesi, tokenizer, Value sınıfı ve SGD yöntemi korunacak.

In [1]:
import random
import math
with open("../data/raw_names.txt", encoding="utf-8") as f:
    docs = [
        line.strip()
        .replace("İ", "i")
        .replace("I", "ı")
        .lower()
        .replace(" ", "")
        for line in f
        if line.strip()
    ]

random.shuffle(docs)

uchars = sorted(set(''.join(docs)))
BOS = len(uchars)
vocab_size = len(uchars) + 1

print("İsim sayısı:", len(docs))
print("Sözlük boyutu:", vocab_size)

İsim sayısı: 1100
Sözlük boyutu: 30


## 3.2 Konum Embeddingleri

Token embeddingi, hangi tokenın işlendiğini temsil eder.
Konum embeddingi ise tokenın dizideki konumunu temsil eder.

Bu iki vektör eleman eleman toplanır. Böylece aynı token,
farklı konumlarda farklı giriş vektörleriyle temsil edilir.
Her iki embedding tablosu da eğitim sırasında öğrenilir.

In [2]:



class Value:
    __slots__ = ('data', 'grad', '_children', '_local_grads')

    def __init__(self, data, children=(), local_grads=()):
        self.data = data
        self.grad = 0
        self._children = children
        self._local_grads = local_grads

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data + other.data, (self, other), (1, 1))

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(
            self.data * other.data,
            (self, other),
            (other.data, self.data)
        )

    def __pow__(self, other):
        return Value(
            self.data ** other,
            (self,),
            (other * self.data ** (other - 1),)
        )

    def log(self):
        return Value(
            math.log(self.data),
            (self,),
            (1 / self.data,)
        )

    def exp(self):
        return Value(
            math.exp(self.data),
            (self,),
            (math.exp(self.data),)
        )

    def relu(self):
        return Value(
            max(0, self.data),
            (self,),
            (float(self.data > 0),)
        )

    def __neg__(self):
        return self * -1

    def __radd__(self, other):
        return self + other

    def __sub__(self, other):
        return self + (-other)

    def __rsub__(self, other):
        return other + (-self)

    def __rmul__(self, other):
        return self * other

    def __truediv__(self, other):
        return self * other ** -1

    def __rtruediv__(self, other):
        return other * self ** -1

    # 2.5 Backward
    def backward(self):
        topo = []
        visited = set()

        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._children:
                    build_topo(child)
                topo.append(v)

        build_topo(self)

        self.grad = 1
        for v in reversed(topo):
            for child, local_grad in zip(v._children, v._local_grads):
                child.grad += local_grad * v.grad

In [3]:

n_embd = 16

block_size = 16

matrix = lambda nout, nin: [
    [Value(random.gauss(0, 0.08)) for _ in range(nin)]
    for _ in range(nout)
]

state_dict = {
    'wte': matrix(vocab_size, n_embd),
    'wpe': matrix(block_size, n_embd),
    'attn_wq': matrix(n_embd, n_embd),
    'attn_wk': matrix(n_embd, n_embd),
    'attn_wv': matrix(n_embd, n_embd),
}

In [4]:
token_id = uchars.index("e")
pos_id = 1

tok_emb = state_dict['wte'][token_id]
pos_emb = state_dict['wpe'][pos_id]

x = [t + p for t, p in zip(tok_emb, pos_emb)]

print("Token embeddingi:", [v.data for v in tok_emb[:3]])
print("Konum embeddingi:", [v.data for v in pos_emb[:3]])
print("Birleşik vektör:", [v.data for v in x[:3]])
print("Vektör uzunluğu:", len(x))

Token embeddingi: [-0.030470541264400578, 0.012987069389232948, 0.034321794097523374]
Konum embeddingi: [-0.07602117623947079, 0.18215511037590318, 0.0865172910499764]
Birleşik vektör: [-0.10649171750387136, 0.19514217976513612, 0.12083908514749978]
Vektör uzunluğu: 16


## 3.3 RMSNorm

RMSNorm, vektördeki değerlerin karelerinin ortalamasını hesaplar.
Bu ortalamanın kareköküne göre bütün değerleri aynı oranda
ölçeklendirir. Sonuçta vektörün RMS değeri yaklaşık 1 olur.

Bu işlem, katmanlara giren değerlerin ölçeğini düzenlemeye
yardımcı olur. Sıfıra bölünmeyi önlemek için küçük bir epsilon
değeri eklenir. LayerNorm'dan farklı olarak ortalama çıkarılmaz.

In [5]:
def rmsnorm(x):
    ms = sum(xi * xi for xi in x) / len(x)
    scale = (ms + 1e-5) ** -0.5
    return [xi * scale for xi in x]

In [6]:
x_norm = rmsnorm(x)

rms_before = (sum(v.data ** 2 for v in x) / len(x)) ** 0.5
rms_after = (
    sum(v.data ** 2 for v in x_norm) / len(x_norm)
) ** 0.5

print("Önceki RMS:", rms_before)
print("Sonraki RMS:", rms_after)
print("İlk 3 normalize değer:", [v.data for v in x_norm[:3]])
print("Vektör uzunluğu:", len(x_norm))

Önceki RMS: 0.11099125771483215
Sonraki RMS: 0.9995943717879067
İlk 3 normalize değer: [-0.959071224621977, 1.7574629624688634, 1.088284536028812]
Vektör uzunluğu: 16


## 3.4 Attention için Q, K ve V

Normalize edilmiş token ve konum vektöründen üç farklı
ağırlık matrisiyle Q, K ve V vektörleri oluşturulur.

Q, mevcut tokenın aradığı bilgiyi temsil eder.
K, tokenların bu aramayla karşılaştırılacak temsilidir.
V ise attention sonucunda aktarılacak bilgileri taşır.

Her üç vektör de 16 elemanlıdır. Bu aşamada yalnızca
vektörler oluşturulur; karşılaştırma sonraki aşamadadır.

In [7]:
def linear(x, w):
    return [sum(wi * xi for wi, xi in zip(wo, x)) for wo in w]

In [8]:
token_id = uchars.index("e")
pos_id = 1

tok_emb = state_dict['wte'][token_id]
pos_emb = state_dict['wpe'][pos_id]
x = [t + p for t, p in zip(tok_emb, pos_emb)]

x_residual = x
x = rmsnorm(x)

q = linear(x, state_dict['attn_wq'])
k = linear(x, state_dict['attn_wk'])
v = linear(x, state_dict['attn_wv'])

print("Q uzunluğu:", len(q))
print("K uzunluğu:", len(k))
print("V uzunluğu:", len(v))

print("Q ilk 3 değer:", [a.data for a in q[:3]])
print("K ilk 3 değer:", [a.data for a in k[:3]])
print("V ilk 3 değer:", [a.data for a in v[:3]])

Q uzunluğu: 16
K uzunluğu: 16
V uzunluğu: 16
Q ilk 3 değer: [0.22821982809240113, 0.24458359733083432, 0.5735109756114954]
K ilk 3 değer: [0.11437851483611731, -0.13559796752170017, -0.3311954248916889]
V ilk 3 değer: [0.30629797164806205, -0.0001296901004074983, 0.13372498698749938]
